# 🔬 Phase 2 — Feature Engineering & Entraînement Risque Retard

Ce notebook explore les données extraites du backend Spring Boot et entraîne le modèle de scoring de risque.

In [ ]:
import sys
sys.path.append('..')  # pour importer app/ depuis notebooks/

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from app.data.extraction import DataExtractor
from app.models.risque_retard import _build_features, _generate_synthetic_target, FEATURE_COLS

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Extraction des données

In [ ]:
extractor = DataExtractor()

df_recettes = extractor.get_recettes()
df_paiements = extractor.get_paiements()
df_proprietaires = extractor.get_proprietaires()
df_avis_tib = extractor.get_avis_tib()
df_avis_tnb = extractor.get_avis_tnb()

print(f'Recettes: {len(df_recettes)} lignes')
print(f'Paiements: {len(df_paiements)} lignes')
print(f'Propriétaires: {len(df_proprietaires)} lignes')
print(f'Avis TIB: {len(df_avis_tib)} lignes')
print(f'Avis TNB: {len(df_avis_tnb)} lignes')

## 2. Feature Engineering

In [ ]:
df_features = _build_features(df_recettes, df_paiements, df_proprietaires, df_avis_tib, df_avis_tnb)
print(f'Features construites: {len(df_features)} propriétaires')
df_features.head(10)

## 3. Distribution des features

In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(20, 12))
axes = axes.flatten()
for i, col in enumerate(FEATURE_COLS):
    ax = axes[i]
    df_features[col].hist(bins=20, ax=ax, color='steelblue', edgecolor='white')
    ax.set_title(col)
plt.tight_layout()
plt.show()

## 4. Cible synthétique (règle métier)

In [ ]:
y = _generate_synthetic_target(df_features)
print('Distribution des classes:')
print(y.value_counts().sort_index())

labels = ['Faible', 'Moyen', 'Élevé', 'Critique']
colors = ['#22c55e', '#eab308', '#f97316', '#ef4444']
plt.bar(labels, y.value_counts().sort_index(), color=colors)
plt.title('Distribution des classes de risque')
plt.ylabel('Nombre de propriétaires')
plt.show()

## 5. Entraînement du modèle

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

X = df_features[FEATURE_COLS].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

model = RandomForestClassifier(
    n_estimators=100,
    max_depth=8,
    random_state=42,
    class_weight='balanced'
)
model.fit(X_train, y_train)

print(f'Train accuracy: {model.score(X_train, y_train):.4f}')
print(f'Test accuracy:  {model.score(X_test, y_test):.4f}')
print('\nCross-validation (5 folds):')
print(cross_val_score(model, X_scaled, y, cv=5))

## 6. Feature Importance

In [ ]:
importance = pd.Series(model.feature_importances_, index=FEATURE_COLS)
importance = importance.sort_values(ascending=True)

plt.figure(figsize=(10, 6))
importance.plot(kind='barh', color='steelblue')
plt.title('Importance des features — RandomForest')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

## 7. Matrice de confusion

In [ ]:
y_pred = model.predict(X_test)
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=labels, yticklabels=labels)
plt.title('Matrice de confusion')
plt.ylabel('Vrai label')
plt.xlabel('Prédiction')
plt.show()

print(classification_report(y_test, y_pred, target_names=labels))

## 8. Sauvegarde du modèle

In [ ]:
import joblib
from pathlib import Path

Path('../models_trained').mkdir(exist_ok=True)
joblib.dump(model, '../models_trained/risque_retard.joblib')
joblib.dump(scaler, '../models_trained/risque_scaler.joblib')
print('✅ Modèle sauvegardé dans models_trained/')

## 9. Test de prédiction individuelle

In [ ]:
from app.models.risque_retard import RisqueRetardModel

# Recharge le modèle fraîchement sauvegardé
m = RisqueRetardModel()

# Prend le premier propriétaire du dataset
pid = int(df_features.iloc[0]['proprietaire_id'])
result = m.predict_for_proprietaire(pid)

print(json.dumps(result, indent=2, ensure_ascii=False))